# Exploring `agent.py` — The Stateful Wrapper

This notebook is a hands-on walkthrough of `liteagent.Agent` — the stateful class that wraps
the raw loop. If the loop notebook explored the **engine**, this one explores the **car**.

**Why two layers?**

The raw loop (`agent_loop`, `agent_loop_continue`) is stateless — you pass in context, it returns
an EventStream, you iterate events yourself. It's the engine.

The `Agent` class wraps the loop and manages:
- **Message history** — accumulates across turns, no manual context threading
- **Event subscription** — `subscribe(callback)` instead of manual `async for`
- **Steering + follow-up queues** — `steer()` and `follow_up()` with dequeue modes
- **Cancellation** — `abort()` with partial message preservation
- **State tracking** — `is_streaming`, `stream_message`, `pending_tool_calls`, `error`

This is the same two-layer design as pi-mono (`agent-loop.ts` + `agent.ts`).
Most consumers will use `Agent`. Advanced consumers may use the raw loop directly.

**What we'll cover:**
1. Setup + imports
2. Simplest prompt — string in, messages out
3. `subscribe()` — the primary consumer API
4. `prompt()` overloads — string, dict, list, images
5. State access — what you can inspect during and after a run
6. Multi-turn — why Agent is stateful
7. Steering — `steer()` mid-run
8. Follow-up — `follow_up()` after idle
9. Queue modes — one-at-a-time vs all
10. `continue_run()` — resume from context
11. `abort()` and partial preservation
12. `wait_for_idle()`
13. `reset()` vs `clear_messages()`
14. Configuration setters — mid-run changes
15. Error handling
16. `_default_convert_to_llm` — what it does
17. Testing across models
18. Real-world patterns
19. Summary

---

## 1. Setup

The Agent needs a model string (litellm format) and optionally tools, system prompt,
and a `convert_to_llm` function. Let's import everything and define our helpers.

In [1]:
from liteagent import Agent, Tool, ToolResult

# Default model for all examples
MODEL = "anthropic/claude-sonnet-4-6"


# convert_to_llm: strips our extras, keeps LLM-compatible fields.
# The Agent provides a default, but we'll define one explicitly so
# we can see exactly what it does (and override for provider quirks later).
def simple_convert(messages):
    result = []
    for m in messages:
        role = m.get("role")
        if role == "assistant":
            msg = {"role": "assistant"}
            if m.get("content"):
                msg["content"] = m["content"]
            if m.get("tool_calls"):
                msg["tool_calls"] = m["tool_calls"]
            if m.get("thinking_blocks"):
                msg["thinking_blocks"] = m["thinking_blocks"]
            if m.get("reasoning_content"):
                msg["reasoning_content"] = m["reasoning_content"]
            result.append(msg)
        elif role == "user":
            result.append({"role": "user", "content": m["content"]})
        elif role == "tool":
            content = m.get("content")
            if isinstance(content, list):
                text_parts = [b["text"] for b in content if b.get("type") == "text"]
                content = "\n".join(text_parts)
            result.append(
                {"role": "tool", "tool_call_id": m["tool_call_id"], "content": content}
            )
    return result


# Simple echo tool — reused across examples
async def echo_execute(tool_call_id, params, signal=None, on_update=None):
    return ToolResult(content=[{"type": "text", "text": params["message"]}])


echo_tool = Tool(
    name="echo",
    description="Echo back a message exactly",
    parameters={
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "The message to echo"}
        },
        "required": ["message"],
    },
    execute=echo_execute,
)

print("Setup complete.")

Setup complete.


## 2. Simplest prompt — string in, messages out

The absolute minimum: create an Agent, call `prompt("...")`, check `agent.messages`.

Unlike the raw loop (where you build `AgentContext` + `AgentConfig`, call `agent_loop`,
and iterate the EventStream yourself), the Agent does all of that internally.
`prompt()` blocks until the loop completes.

In [2]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence max.",
    convert_to_llm=simple_convert,
)

await agent.prompt("What is 2 + 2?")



In [3]:
agent.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772840701289},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772840701977}]

In [4]:
agent.state.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772840701289},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772840701977}]

In [5]:
agent.state

AgentState(system_prompt='Be concise. One sentence max.', model='anthropic/claude-sonnet-4-6', thinking_level='off', tools=[], messages=[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772840701289}, {'role': 'assistant', 'content': '4', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 25, 'completion_tokens': 5, 'total_tokens': 30, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'timestamp': 1772840701977}], is_streaming=False, stream_message=None, pending_tool_calls=set(), error=None)

In [6]:
agent.state.messages

[{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772840701289},
 {'role': 'assistant',
  'content': '4',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 25,
   'completion_tokens': 5,
   'total_tokens': 30,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772840701977}]

In [7]:
simple_convert(agent.state.messages)

[{'role': 'user', 'content': 'What is 2 + 2?'},
 {'role': 'assistant', 'content': '4'}]

Two messages: the user prompt we sent, and the assistant's reply.
The Agent appended both to `agent.messages` automatically —
this is the key difference from the raw loop where you manage context yourself.

Let's look at the raw message objects:

In [8]:
# User message — our input, wrapped in a dict by prompt()
agent.messages[0]

{'role': 'user', 'content': 'What is 2 + 2?', 'timestamp': 1772840701289}

In [9]:
# Assistant message — enriched with usage, stop_reason, timestamp
agent.messages[1]

{'role': 'assistant',
 'content': '4',
 'tool_calls': None,
 'thinking_blocks': None,
 'reasoning_content': None,
 'provider_specific_fields': None,
 'usage': {'prompt_tokens': 25,
  'completion_tokens': 5,
  'total_tokens': 30,
  'cache_read_tokens': 0,
  'cache_creation_tokens': 0},
 'stop_reason': 'stop',
 'timestamp': 1772840701977}

In [10]:
# This is why we usd @property --> Read-only access. With @property, ust prevents replacing the state object itself.
agent.state = 'BREAK THE STATE'

AttributeError: property 'state' of 'Agent' object has no setter

The assistant message has all the extras the loop adds:
- `usage` — token counts from litellm
- `stop_reason` — "stop" (normal), "tool_calls", "error", "aborted"
- `timestamp` — Unix ms
- `thinking_blocks` / `reasoning_content` — None unless thinking is enabled
- `provider_specific_fields` — opaque bag from litellm

These extras are why `convert_to_llm` exists — they must be stripped before
sending messages back to the LLM.

## 3. `subscribe()` — the primary consumer API

In the loop notebook, we used `async for event in stream` to consume events.
The Agent doesn't expose the stream. Instead, you subscribe a callback:

```python
unsub = agent.subscribe(my_callback)  # returns unsubscribe function
```

The callback fires synchronously during `await agent.prompt()` — same thread,
no concurrency issues. This is how pi's agent works too.

**Why callbacks instead of async iteration?** The Agent is the sole reader of
the loop's EventStream (internal detail). External consumers get events via
subscribe — this lets multiple consumers see the same events (unlike a queue
where each item is consumed once).

In [ ]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)

# Collect all events
events = []
unsub = agent.subscribe(lambda e: events.append(e))

await agent.prompt("Say hello!")


In [ ]:
for e in events:
    print(e)

Same events sequence as the raw loop:

Now let's test unsubscribe:

In [ ]:
count_before = len(events)
count_before

In [ ]:
agent._subscribers

In [ ]:
unsub()  # stop receiving events

In [ ]:
agent._subscribers

In [ ]:
await agent.prompt("Say goodbye.")

count_after = len(events)
print(f"Events before unsub: {count_before}")
print(f"Events after second prompt: {count_after}")
print(f"Unsubscribe worked: {count_before == count_after}")

In [ ]:
agent.state.messages

## 4. `prompt()` overloads

Like pi's `agent.prompt()`, ours accepts four input shapes:

| Input | What happens |
|-------|-------------|
| `prompt("string")` | Wrapped in `{"role": "user", "content": "string", "timestamp": ...}` |
| `prompt({"role": "user", ...})` | Used as-is |
| `prompt([msg1, msg2])` | Multiple messages injected |
| `prompt("text", images=[...])` | Multimodal: text + images in content array |

Let's see what each produces.

In [ ]:
# Overload 1: string
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
await agent.prompt("Hello from a string")
agent.messages[0]

In [ ]:
# Overload 2: dict (used as-is)
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
await agent.prompt(
    {"role": "user", "content": "Hello from a dict", "custom_field": "preserved"}
)
agent.messages[0]

In [ ]:
simple_convert(agent.messages)

In [ ]:
# Overload 3: list of messages
agent = Agent(model=MODEL, convert_to_llm=simple_convert, system_prompt="Be concise.")
await agent.prompt(
    [
        {"role": "user", "content": "My name is Alice."},
        {"role": "user", "content": "What is my name?"},
    ]
)

In [ ]:
agent.messages

In [ ]:
# Overload 4: string + images (multimodal)
# Send a real image and ask the LLM about it

image_block = {
    "type": "image_url",
    "image_url": {"url": f"https://dev-dashhudson-static.s3.amazonaws.com/research/media_asset_ai_generation/experiments/urbn_flat_lay/92207174_707_b3.jpg"},
}

agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence max.",
    convert_to_llm=simple_convert,
)
await agent.prompt("What is in this image?", images=[image_block])

print(f"Response: {agent.messages[-1].get('content')}")

In [ ]:
agent.messages

## 5. State access

The Agent tracks state in an `AgentState` dataclass. You can inspect it at any time:

```python
agent.state.is_streaming       # True while loop is running
agent.state.stream_message     # current partial message being streamed (or None)
agent.state.pending_tool_calls # set of tool call IDs currently executing
agent.state.error              # last error message (or None)
agent.state.model              # current model string
agent.state.system_prompt      # current system prompt
agent.state.tools              # current tool list
agent.state.thinking_level     # "off", "minimal", "low", "medium", "high", "xhigh"
agent.messages                 # shorthand for agent.state.messages
```

Let's watch state change *during* a run using a subscriber:

In [ ]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
    tools=[echo_tool],
)

# Track state transitions
state_log = []

print(
    f"{'Event':<26} {'Streaming':<10} {'StreamMsg':<10} {'PendTools':<10} {'MsgCount':<10}"
)
def track_state(event):
    t = event["type"]
    entry = {
        "event": t,
        "is_streaming": agent.state.is_streaming,
        "stream_msg": agent.state.stream_message is not None,
        "pending_tools": len(agent.state.pending_tool_calls),
        "msg_count": len(agent.messages),
    }
    state_log.append(entry)
    print("-" * 66)
    print(
            f"{entry['event']:<26} {str(entry['is_streaming']):<10} {str(entry['stream_msg']):<10} {entry['pending_tools']:<10} {entry['msg_count']:<10}"
        )

agent.subscribe(track_state)
await agent.prompt("Echo 'hello world'")



In [ ]:
import pandas as pd
pd.DataFrame(state_log)

Notice how:
- `is_streaming` is True throughout the run
- `stream_message` appears on `message_start` (assistant only), disappears on `message_end`
- `pending_tool_calls` increments on `tool_execution_start`, decrements on `tool_execution_end`
- `msg_count` grows on each `message_end` — messages are appended incrementally, not batched


The tool call produces this pattern:
```
Turn 1: assistant calls echo → tool executes → tool result
Turn 2: assistant sees result → responds with text
```

Messages: user → assistant (tool_call) → tool (result) → assistant (text response)

## 6. Multi-turn — why Agent is stateful

This is the Agent's main value: messages persist across `prompt()` calls.
With the raw loop, you'd need to manually thread context between calls.
The Agent does it automatically.

In [ ]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. Remember everything the user says.",
    convert_to_llm=simple_convert,
)

# Turn 1: tell the agent something
await agent.prompt("My favorite color is blue.")
agent.messages



In [ ]:
# Turn 2: ask about it — the agent should remember
await agent.prompt("What is my favorite color?")
agent.messages

## 7. Steering — `steer()` mid-run

`steer()` queues a message that gets injected **during** a run:
- After each tool execution, the loop checks the steering queue
- If there's a message, remaining tools are **skipped** and the steering message
  is injected before the next LLM call
- This is "stop what you're doing, do this instead"

The loop also checks for steering at the start of each run (before the first LLM call).
So if you call `steer()` before `prompt()`, the steering message gets picked up immediately.

Let's demonstrate both: pre-queued steering, and mid-tool steering.

In [ ]:
# Pre-queued steering: steer() before prompt()
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)

agent.steer("Actually, tell me a joke instead.")
await agent.prompt("What is the capital of France?")

agent.messages

In [ ]:
# Mid-tool steering: steer() during tool execution
# When tool_a executes, it queues a steering message.
# tool_b should be SKIPPED.

call_log = []
steering_agent = None  # forward reference


async def tool_a_exec(tool_call_id, params, signal=None, on_update=None):
    call_log.append("a")
    steering_agent.steer("Stop! Do something else.")  # interrupt!
    return ToolResult(content=[{"type": "text", "text": "tool_a done"}])


async def tool_b_exec(tool_call_id, params, signal=None, on_update=None):
    call_log.append("b")
    return ToolResult(content=[{"type": "text", "text": "tool_b done"}])


tool_a = Tool(
    name="tool_a",
    description="Tool A",
    parameters={"type": "object", "properties": {}},
    execute=tool_a_exec,
)
tool_b = Tool(
    name="tool_b",
    description="Tool B",
    parameters={"type": "object", "properties": {}},
    execute=tool_b_exec,
)

steering_agent = Agent(
    model=MODEL,
    system_prompt="When asked, call both tool_a and tool_b in a single response. Be concise.",
    convert_to_llm=simple_convert,
    tools=[tool_a, tool_b],
)

await steering_agent.prompt("Call both tool_a and tool_b now.")

steering_agent.messages

  1. Assistant calls both tool_a and tool_b
  2. tool_a executes (and queues steering inside its execute function)
  3. tool_b gets skipped — is_error: True, "Skipped due to queued user message."
  4. Steering message injected: "Stop! Do something else."
  5. Assistant responds to the steering instead of continuing

  The key proof: tool_b never ran ('b' not in call_log), but it still has a tool result in the conversation — that's the synthetic skip result from _skip_tool_call() in
  loop.py:94-129. The LLM needs every tool call to have a result, even skipped ones.

## 8. Follow-up — `follow_up()` after idle

`follow_up()` is the *outer loop* mechanism. Unlike steering (which interrupts),
follow-ups wait until the agent finishes everything (no more tool calls, no steering).
Then the follow-up message is injected and the agent continues.

- Steering = "stop what you're doing" (immediate)
- Follow-up = "when you're done, also do this" (deferred)

In [ ]:
agent = Agent(
    model=MODEL,
    system_prompt="Be concise. One sentence.",
    convert_to_llm=simple_convert,
)

# Queue a follow-up BEFORE the first prompt
agent.follow_up("Now tell me a fun fact about cats.")

await agent.prompt("What is 2 + 2?")

agent.messages

## 9. Queue modes

Both steering and follow-up have two modes:
- `"one-at-a-time"` (default) — dequeue one message per poll
- `"all"` — dequeue everything at once

This matters when multiple messages are queued. Let's see the difference.

In [ ]:
# one-at-a-time (default): queue 3, dequeue returns 1
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
agent.steer("msg1")
agent.steer("msg2")
agent.steer("msg3")

batch = agent._dequeue_steering()
print(
    f"one-at-a-time: got {len(batch)} message(s), {len(agent._steering_queue)} remaining"
)
print(f"  dequeued: '{batch[0]['content']}'")

In [ ]:
# all mode: queue 3, dequeue returns all 3
agent = Agent(model=MODEL, convert_to_llm=simple_convert, steering_mode="all")
agent.steer("msg1")
agent.steer("msg2")
agent.steer("msg3")

batch = agent._dequeue_steering()
print(f"all mode: got {len(batch)} message(s), {len(agent._steering_queue)} remaining")
for m in batch:
    print(f"  '{m['content']}'")

## 10. `continue_run()` — resume from context

`continue_run()` is for when the conversation ended at a tool result or user message
and you want the LLM to continue from there — without sending a new prompt.

Three interesting cases when the last message is an assistant message:
1. Steering queue has messages → use those
2. Follow-up queue has messages → use those
3. Both empty → error (can't continue from assistant without new input)

**When would you actually use this?** In normal chat (`prompt()` → response → `prompt()` again),
you won't. `continue_run()` is for **recovery and resumption** — when something outside the
normal flow modifies the message history. Real-world examples from pi-mono's coding agent:

- **Context compaction** — when the conversation gets too long, old messages are summarized and
  replaced. After compaction the last message might be a tool result the LLM never responded to.
  `continue()` kicks the loop to pick up from there.
- **Error retry** — when an LLM call fails, a "please retry" message is appended and `continue()`
  re-runs the loop without needing a new user prompt.
- **Queued messages after idle** — if `follow_up()` or `steer()` is called after the agent
  finishes, `continue_run()` restarts the loop to process them.

In [15]:
# Case: continue from a tool result (manually built context)
agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)

# Simulate: user asked about weather → assistant called tool → we have the result
agent._state.messages = [
    {"role": "user", "content": "What's the weather?"},
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {"id": "c0", "type": "function", "function": {"name": "weather", "arguments": "{}"}}
        ],
        "stop_reason": "tool_calls",
    },
    {
        "role": "tool",
        "tool_call_id": "c0",
        "content": [{"type": "text", "text": "72°F and sunny in San Francisco"}],
        "is_error": False,
    },
]

await agent.continue_run()

agent.messages

[{'role': 'user', 'content': "What's the weather?"},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'c0',
    'type': 'function',
    'function': {'name': 'weather', 'arguments': '{}'}}],
  'stop_reason': 'tool_calls'},
 {'role': 'tool',
  'tool_call_id': 'c0',
  'content': [{'type': 'text', 'text': '72°F and sunny in San Francisco'}],
  'is_error': False},
 {'role': 'assistant',
  'content': "I don't have a **weather tool** available to check current conditions. Could you please:\n\n1. **Share your location** (city, zip code, etc.), and\n2. Try a weather service like [weather.com](https://www.weather.com), a search engine, or a virtual assistant on your device for real-time updates?\n\nI'm sorry I can't pull that directly for you!",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 609,
   'completion_tokens': 90,
   'total_tokens': 699,
   'cache_read_tokens': 0,
   'cache_

In [21]:
# Case: continue from assistant + steering queue
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
agent._state.messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hello!", "stop_reason": "stop"},
]
agent.steer("Now tell me a joke.")

await agent.continue_run()

agent.messages

[{'role': 'user', 'content': 'Hi'},
 {'role': 'assistant', 'content': 'Hello!', 'stop_reason': 'stop'},
 {'role': 'user',
  'content': 'Now tell me a joke.',
  'timestamp': 1772842210188},
 {'role': 'assistant',
  'content': "Sure! Here's one:\n\nWhy don't scientists trust atoms?\n\nBecause they make up everything! 😄\n\nWant to hear another one?",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 22,
   'completion_tokens': 34,
   'total_tokens': 56,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772842211472}]

In [22]:
# Case: continue from assistant + empty queues → error
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
agent._state.messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hello!", "stop_reason": "stop"},
]

try:
    await agent.continue_run()
except ValueError as e:
    print(f"Got expected error: {e}")

Got expected error: Cannot continue from assistant message without queued messages. Use steer() or follow_up() first.


## 11. `abort()` and partial preservation

`abort()` sets a signal that stops the loop. But what happens to the partial
assistant message that was being streamed? The Agent handles this edge case
(same as pi's agent.ts lines 504-518):

1. If the partial has **real content** (non-empty text, reasoning, or named tool call) → **preserve it**
2. If it's just **empty scaffolding** (empty strings, unnamed tool calls) → **discard it**
3. If discarded after abort → raise "Request was aborted" → caught by error handler

In [41]:
# abort() after receiving some text — partial should be preserved
agent = Agent(
    model=MODEL,
    system_prompt="Write a very long essay about the history of computing. At least 5000 words.",
    convert_to_llm=simple_convert,
)

chunk_count = 0


def abort_after_chunks(event):
    print('signal is set: ' + str(agent._signal.is_set()))
    if agent.state.stream_message:
        print(agent.state.stream_message['content'])
    global chunk_count
    if event["type"] == "message_update" and event.get("delta_type") == "text_delta":
        chunk_count += 1
        if chunk_count >= 5:
            agent.abort()


agent.subscribe(abort_after_chunks)
await agent.prompt("Go ahead.")

print(f"is_streaming: {agent.state.is_streaming}")
print(f"signal cleaned up: {agent._signal is None}")


signal is set: False
signal is set: False
signal is set: False
signal is set: False
signal is set: False
None
signal is set: False
# The History of Computing: From
signal is set: False
# The History of Computing: From Ancient Ab
signal is set: False
# The History of Computing: From Ancient Abacus to
signal is set: False
# The History of Computing: From Ancient Abacus to Artificial
signal is set: False
# The History of Computing: From Ancient Abacus to Artificial Intelligence

## A
signal is set: True
# The History of Computing: From Ancient Abacus to Artificial Intelligence

## A Comprehensive Survey
signal is set: True
signal is set: True
signal is set: True
is_streaming: False
signal cleaned up: True


In [42]:
agent.messages

[{'role': 'user', 'content': 'Go ahead.', 'timestamp': 1772844367217},
 {'role': 'assistant',
  'content': '# The History of Computing: From Ancient Abacus to Artificial Intelligence\n\n## A Comprehensive Survey',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 0,
   'completion_tokens': 22,
   'total_tokens': 22,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'aborted',
  'timestamp': 1772844368357}]

## 12. `wait_for_idle()`

  A coordination primitive. `prompt()` already awaits internally, so when it returns
  the agent is idle. `wait_for_idle()` is for when something *else* triggered the agent
  and you need to sync from a different place in your code:

  ```python
  # Some callback triggered a run
  agent.follow_up("do something")
  asyncio.create_task(agent.continue_run())

  # Later, elsewhere:
  await agent.wait_for_idle()   # block until that run finishes
  # now safe to inspect agent.messages

  If you're always doing await agent.prompt() sequentially, you'll never need this.

In [44]:
# When idle, returns immediately
agent = Agent(model=MODEL, convert_to_llm=simple_convert)
await agent.wait_for_idle()  # should not hang
print("wait_for_idle() returned immediately (agent is idle)")

# After a prompt, also returns immediately (prompt already blocks)
await agent.prompt("Hi")
await agent.wait_for_idle()
print("wait_for_idle() returned immediately (prompt already completed)")

wait_for_idle() returned immediately (agent is idle)
wait_for_idle() returned immediately (prompt already completed)


## 13. `reset()` vs `clear_messages()`

Two ways to clear state, with different scopes:

| Method | Clears messages | Clears queues | Clears error | Keeps config |
|--------|:-:|:-:|:-:|:-:|
| `reset()` | ✓ | ✓ | ✓ | ✓ |
| `clear_messages()` | ✓ | ✗ | ✗ | ✓ |

`reset()` is "start over". `clear_messages()` is "clear history but keep queued work".

In [47]:
agent = Agent(
    model=MODEL,
    system_prompt="you are a nice agent",
    tools=[echo_tool],
    convert_to_llm=simple_convert,
)
agent.append_message({"role": "user", "content": "old message"})
agent.steer("queued steering")
agent.follow_up("queued follow-up")
agent._state.error = "some error"

print("Before clear_messages():")
agent.messages



Before clear_messages():


[{'role': 'user', 'content': 'old message'}]

In [48]:
agent.clear_messages()

print("After clear_messages():")
agent.messages

After clear_messages():


[]

In [49]:
# Now reset — clears everything
agent.append_message({"role": "user", "content": "new message"})
agent.reset()

print("After reset():")
print(
    f"  messages: {len(agent.messages)}, queued: {agent.has_queued_messages()}, error: {agent.state.error}"
)
print(f"  model: {agent.state.model}  ← preserved")
print(f"  system_prompt: '{agent.state.system_prompt}'  ← preserved")
print(f"  tools: {len(agent.state.tools)}  ← preserved")

After reset():
  messages: 0, queued: False, error: None
  model: anthropic/claude-sonnet-4-6  ← preserved
  system_prompt: 'you are a nice agent'  ← preserved
  tools: 1  ← preserved


## 14. Configuration setters — mid-run changes

Pi allows calling `setModel()`, `setTools()`, etc. even while the agent is streaming.
The loop snapshots context at the start of each run, so mid-run changes only take
effect on the **next** run. We match this behavior — no streaming guard.

This enables patterns like:
- **Dynamic capabilities:** escalate tool permissions mid-conversation
- **Model switching:** use a cheap model for simple tasks, expensive for complex ones
- **Plan mode:** swap tool sets between planning and execution phases

In [55]:
agent = Agent(model=MODEL, convert_to_llm=simple_convert)

# Track which model gets called
def event_handler(e):
    if e['type'] == 'message_end':
        print(f'Model Used: {agent.state.model}')
agent.subscribe(event_handler)

await agent.prompt("hey")

# Switch model mid-conversation
agent.set_model("gemini/gemini-3-flash-preview")

await agent.prompt("Bye")
agent.messages

Model Used: anthropic/claude-sonnet-4-6
Model Used: anthropic/claude-sonnet-4-6
Model Used: gemini/gemini-3-flash-preview
Model Used: gemini/gemini-3-flash-preview


[{'role': 'user', 'content': 'hey', 'timestamp': 1772845997628},
 {'role': 'assistant',
  'content': "Hey! How's it going? What's on your mind? 😊",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 8,
   'completion_tokens': 19,
   'total_tokens': 27,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772845998635},
 {'role': 'user', 'content': 'Bye', 'timestamp': 1772845998636},
 {'role': 'assistant',
  'content': 'Goodbye! Have a great rest of your day. Feel free to reach out if you need anything later!',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': {'thought_signatures': ['EjQKMgG+Pvb7J1W8Hnas+KWl5DsRL1xkM+znpS3W/z4YjWi4VDv/pCVSPfQRnJRi9uPFvpzE']},
  'usage': {'prompt_tokens': 21,
   'completion_tokens': 21,
   'total_tokens': 42,
   'cache_read_tokens': 0,
   'cache_creation_token

## 15. Error handling

When the LLM call fails (network error, rate limit, etc.), the Agent:
1. Catches the exception
2. Creates a synthetic assistant message with `stop_reason="error"`
3. Appends it to messages
4. Sets `agent.state.error`
5. Emits `agent_end` event
6. Cleans up (is_streaming=False, etc.)

The Agent does **not** re-raise — it always completes cleanly. This lets consumers
check `agent.state.error` instead of wrapping every `prompt()` in try/except.

In [ ]:
# Force an error by using a non-existent model
agent = Agent(model="fake-provider/nonexistent-model", convert_to_llm=simple_convert)
await agent.prompt("This will fail.")

agent.messages


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



[{'role': 'user', 'content': 'This will fail.', 'timestamp': 1772846378217},
 {'role': 'assistant',
  'content': None,
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'model': 'fake-provider/nonexistent-model',
  'usage': {'prompt_tokens': 0,
   'completion_tokens': 0,
   'total_tokens': 0,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'error',
  'error_message': "litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=fake-provider/nonexistent-model\n Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers",
  'timestamp': 1772846378229}]

## 16. `_default_convert_to_llm`

If you don't provide `convert_to_llm`, the Agent uses a built-in default.
Let's see what it does vs what we'd need for specific providers.

In [57]:
from liteagent.agent import _default_convert_to_llm

# Simulate a conversation with enriched messages
messages = [
    {"role": "user", "content": "Hi", "timestamp": 12345},
    {
        "role": "assistant",
        "content": "Hello!",
        "tool_calls": None,
        "thinking_blocks": None,
        "reasoning_content": None,
        "usage": {"prompt_tokens": 10, "completion_tokens": 5},
        "stop_reason": "stop",
        "timestamp": 12346,
        "provider_specific_fields": {"some": "thing"},
    },
    {
        "role": "tool",
        "tool_call_id": "c0",
        "content": [{"type": "text", "text": "result"}],
        "name": "echo",
        "is_error": False,
        "details": {"extra": "data"},
        "timestamp": 12347,
    },
]

converted = _default_convert_to_llm(messages)
converted


[{'role': 'user', 'content': 'Hi'},
 {'role': 'assistant', 'content': 'Hello!'},
 {'role': 'tool', 'tool_call_id': 'c0', 'content': 'result'}]

The default `convert_to_llm` is good enough for most cases. You'd override it when:

1. **Custom message types** — your app stores `{"role": "notification", ...}` that need
   to be filtered out or converted to user messages
2. **Provider quirks** — OpenAI requires tool result content as a plain string, not
   a content block array (the default already handles this for tool messages)
3. **Multimodal tool results** — images in tool results need to be routed to a
   follow-up user message for providers that don't support images in tool results

## 17. Testing across models

The Agent is model-agnostic. Let's run the same prompt + tool call through
all 4 active target models.

In [58]:
MODELS = [
    "anthropic/claude-sonnet-4-6",
    "anthropic/claude-opus-4-6",
    "gemini/gemini-3-flash-preview",
    "gpt-5.2",
]

for model in MODELS:
    agent = Agent(
        model=model,
        system_prompt="Use the echo tool. Be concise.",
        convert_to_llm=simple_convert,
        tools=[echo_tool],
    )

    try:
        await agent.prompt("Echo 'test'")
        roles = [m.get("role") for m in agent.messages]
        has_tool = "tool" in roles
        assistants = [m for m in agent.messages if m.get("role") == "assistant"]
        last_content = (assistants[-1].get("content") or "")[:50] if assistants else "?"
        print(f"  ✓ {model:<42} tool_used={has_tool} | {last_content}")
    except Exception as e:
        print(f"  ✗ {model:<42} ERROR: {e}")

  ✓ anthropic/claude-sonnet-4-6                tool_used=True | The echo tool returned: **test**
  ✓ anthropic/claude-opus-4-6                  tool_used=True | Done! The message "test" was echoed back.
  ✓ gemini/gemini-3-flash-preview              tool_used=True | test
  ✓ gpt-5.2                                    tool_used=True | test


## 18. Real-world patterns

The Agent is framework-agnostic. Here's how you'd wire it into different consumers.

In [61]:
# Pattern 1: CLI — print text deltas as they arrive

agent = Agent(
    model=MODEL,
    system_prompt="Be concise.",
    convert_to_llm=simple_convert,
)


def cli_handler(event):
    if event["type"] == "message_update" and event.get("delta_type") == "text_delta":
        text = event["delta"].get("content", "")
        print(text, end="", flush=True)
    elif event["type"] == "agent_end":
        print()  # newline at the end


agent.subscribe(cli_handler)
print("Agent: ", end="")
await agent.prompt("What is the meaning of life, in one sentence?")

Agent: The meaning of life is whatever purpose, connection, and fulfillment you consciously choose to create and pursue.
